In [2]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [3]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

from causal_opt.fmri_data import (
    load_fmri_state_data,
    load_paired_tsv_states,
)

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
FMRI_ROOT = Path(
    os.getenv(
        "FMRI_CONNECTIVITY_ROOT",
        ROOT.parent / "fmri_connectivity",
    )
)

# Original CONN BN19 outputs
CONN_ROOT = (
    FMRI_ROOT
    / "data"
    / "mat_files"
    / "rs_sessions_r03_healthy"
)

CONN_SESSION1 = CONN_ROOT / "roi_rs_sessions_Session1.zip"
CONN_SESSION2 = CONN_ROOT / "roi_rs_sessions_Session2.zip"

# New fMRIPrep-derived BN19 outputs
BN19_TS_ROOT = (
    FMRI_ROOT
    / "data"
    / "derivatives"
    / "r03_healthy"
    / "rois"
    / "BN19"
)

BN19_LABELS = (
    FMRI_ROOT
    / "data"
    / "templates"
    / "ROIs"
    / "BN19_labels.json"
)

# ---------------------------------------------------------
# Load both pipelines
# ---------------------------------------------------------
conn_control, conn_sdv = load_fmri_state_data(
    CONN_SESSION1,
    CONN_SESSION2,
)

fmriprep_control, fmriprep_sdv = load_paired_tsv_states(
    BN19_TS_ROOT,
    BN19_LABELS,
    pairing="intersection",
)

# ---------------------------------------------------------
# Match BN19 ROIs using MNI coordinates
# ---------------------------------------------------------
D = np.linalg.norm(
    conn_control.roi_xyz[:, None, :]
    - fmriprep_control.roi_xyz[None, :, :],
    axis=2,
)

conn_idx, prep_idx = linear_sum_assignment(D)

# Put matching fMRIPrep columns into CONN order
order = prep_idx[np.argsort(conn_idx)]

matched_distances = D[conn_idx, prep_idx]

print(
    "ROI coordinate matching:"
    f" median={np.median(matched_distances):.3f} mm,"
    f" max={np.max(matched_distances):.3f} mm"
)

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def zscore_columns(X):
    X = np.asarray(X, float)
    std = X.std(axis=0, ddof=1)
    return (X - X.mean(axis=0)) / std


def align_with_lag(A, B, lag):
    """
    Compare CONN A[t] with fMRIPrep B[t + lag].
    Positive lag = fMRIPrep shifted later.
    """
    if lag >= 0:
        n = min(len(A), len(B) - lag)
        return A[:n], B[lag:lag+n]
    else:
        lag = -lag
        n = min(len(A) - lag, len(B))
        return A[lag:lag+n], B[:n]


def compare_subject_state(conn_state, prep_state, subject, max_lag=5):

    A = np.asarray(conn_state.timeseries[subject], float)
    B = np.asarray(prep_state.timeseries[subject], float)[:, order]

    print(
        f"{subject}: CONN shape={A.shape}, "
        f"fMRIPrep shape={B.shape}"
    )

    # Search a small lag range in case dummy-volume handling differs
    lag_scores = {}

    for lag in range(-max_lag, max_lag + 1):
        a, b = align_with_lag(A, B, lag)

        za = zscore_columns(a)
        zb = zscore_columns(b)

        roi_r = np.array([
            np.corrcoef(za[:, j], zb[:, j])[0, 1]
            for j in range(za.shape[1])
        ])

        lag_scores[lag] = np.median(roi_r)

    best_lag = max(lag_scores, key=lag_scores.get)

    A2, B2 = align_with_lag(A, B, best_lag)
    ZA = zscore_columns(A2)
    ZB = zscore_columns(B2)

    roi_r = np.array([
        np.corrcoef(ZA[:, j], ZB[:, j])[0, 1]
        for j in range(ZA.shape[1])
    ])

    # Compare functional-connectivity structure too
    FC_A = np.corrcoef(ZA, rowvar=False)
    FC_B = np.corrcoef(ZB, rowvar=False)

    upper = np.triu_indices_from(FC_A, k=1)

    fc_r = np.corrcoef(
        FC_A[upper],
        FC_B[upper],
    )[0, 1]

    # Raw amplitude difference
    std_ratio = np.median(
        B2.std(axis=0, ddof=1)
        / A2.std(axis=0, ddof=1)
    )

    roi_table = pd.DataFrame({
        "CONN ROI": np.asarray(conn_state.roi_names),
        "fMRIPrep ROI": np.asarray(prep_state.roi_names)[order],
        "MNI distance mm": matched_distances,
        "timeseries r": roi_r,
        "CONN std": A2.std(axis=0, ddof=1),
        "fMRIPrep std": B2.std(axis=0, ddof=1),
    })

    summary = {
        "subject": subject,
        "best_lag_TR": best_lag,
        "median_ROI_r": np.median(roi_r),
        "min_ROI_r": np.min(roi_r),
        "max_ROI_r": np.max(roi_r),
        "FC_matrix_r": fc_r,
        "median_std_ratio_fmriprep_over_CONN": std_ratio,
    }

    return summary, roi_table, lag_scores


# ---------------------------------------------------------
# First quick check: Subject002 control
# ---------------------------------------------------------
summary, roi_table, lag_scores = compare_subject_state(
    conn_control,
    fmriprep_control,
    "Subject002",
)

display(pd.Series(summary))
display(
    roi_table.sort_values("timeseries r")
)

print("Lag search:")
display(pd.Series(lag_scores, name="median ROI correlation"))

ROI coordinate matching: median=0.000 mm, max=0.000 mm
Subject002: CONN shape=(192, 19), fMRIPrep shape=(192, 19)


/home/ab126/projects/CausalOpt/src/causal_opt/fmri_data.py:228: UserWarning: Retaining 18 paired subjects; excluded control=(), SDV=('Subject015',) (opposite state missing).
  return pair_fmri_states(control, sdv, pairing=pairing)


subject                                Subject002
best_lag_TR                                    -2
median_ROI_r                             0.019726
min_ROI_r                               -0.183196
max_ROI_r                                0.159815
FC_matrix_r                             -0.081942
median_std_ratio_fmriprep_over_CONN      2.998046
dtype: object

,CONN ROI,fMRIPrep ROI,MNI distance mm,timeseries r,CONN std,fMRIPrep std
14,Bladder Network 19.cluster015,Inferior Frontal Gyrus,0.0,-0.183196,0.639259,7.299212
15,Bladder Network 19.cluster016,Left Insula,0.0,-0.053393,0.831997,2.213399
10,Bladder Network 19.cluster011,Pontine Micturition Center 2,0.0,-0.051053,2.833357,3.070502
9,Bladder Network 19.cluster010,Pontine Micturition Center 1,0.0,-0.045128,2.283242,20.906025
4,Bladder Network 19.cluster005,Dorsolateral Prefrontal Cortex 2,0.0,-0.018606,0.908053,8.049727
11,Bladder Network 19.cluster012,Supplementary Motor Area 1,0.0,0.001889,0.597334,3.602048
0,Bladder Network 19.cluster001,Cerebellum 1,0.0,0.013431,1.632384,4.728320
16,Bladder Network 19.cluster017,Medial Prefrontal Cortex,0.0,0.014029,1.679105,11.828135
3,Bladder Network 19.cluster004,Dorsolateral Prefrontal Cortex 1,0.0,0.019527,0.960338,4.885042
7,Bladder Network 19.cluster008,Periaqueductal Gray 2,0.0,0.019726,3.285568,4.923429


Lag search:


-5   -0.003771
-4   -0.015906
-3   -0.018900
-2    0.019726
-1    0.012888
 0    0.003082
 1    0.007762
 2    0.009970
 3   -0.015400
 4   -0.009342
 5   -0.017832
Name: median ROI correlation, dtype: float64

In [4]:
rows = []

for state_name, conn_state, prep_state in [
    ("Control", conn_control, fmriprep_control),
    ("SDV", conn_sdv, fmriprep_sdv),
]:
    common = sorted(
        set(conn_state.subject_ids)
        & set(prep_state.subject_ids)
    )

    for subject in common:
        summary, _, _ = compare_subject_state(
            conn_state,
            prep_state,
            subject,
            max_lag=5,
        )
        summary["state"] = state_name
        rows.append(summary)

comparison = pd.DataFrame(rows)

display(
    comparison[
        [
            "state",
            "subject",
            "best_lag_TR",
            "median_ROI_r",
            "min_ROI_r",
            "max_ROI_r",
            "FC_matrix_r",
            "median_std_ratio_fmriprep_over_CONN",
        ]
    ]
)

print("\nSummary by state:")
display(
    comparison.groupby("state")[
        [
            "median_ROI_r",
            "FC_matrix_r",
            "median_std_ratio_fmriprep_over_CONN",
        ]
    ].agg(["median", "mean", "min", "max"])
)

Subject002: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject003: CONN shape=(194, 19), fMRIPrep shape=(192, 19)
Subject004: CONN shape=(192, 19), fMRIPrep shape=(194, 19)
Subject005: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject006: CONN shape=(193, 19), fMRIPrep shape=(192, 19)
Subject007: CONN shape=(192, 19), fMRIPrep shape=(193, 19)
Subject008: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject009: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject010: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject011: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject012: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject013: CONN shape=(168, 19), fMRIPrep shape=(192, 19)
Subject014: CONN shape=(192, 19), fMRIPrep shape=(168, 19)
Subject016: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject017: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject018: CONN shape=(192, 19), fMRIPrep shape=(192, 19)
Subject019: CONN shape=(192, 19), fMRIPrep shape=(192, 1

,state,subject,best_lag_TR,median_ROI_r,min_ROI_r,max_ROI_r,FC_matrix_r,median_std_ratio_fmriprep_over_CONN
0,Control,Subject002,-2,0.019726,-0.183196,0.159815,-0.081942,2.998046
1,Control,Subject003,1,0.043725,-0.084892,0.167513,0.039261,3.814449
2,Control,Subject004,0,0.029835,-0.158254,0.121302,-0.039006,2.478084
3,Control,Subject005,-2,0.038395,-0.106668,0.165547,0.004103,2.191149
4,Control,Subject006,1,0.027793,-0.125917,0.082589,0.106230,3.267723
5,Control,Subject007,1,-0.001961,-0.108600,0.118941,0.172583,2.617164
6,Control,Subject008,1,0.006239,-0.157074,0.134009,0.108090,3.283804
7,Control,Subject009,-3,0.018593,-0.146829,0.137593,0.094265,2.117703
8,Control,Subject010,-3,0.010607,-0.136949,0.167180,0.217266,2.605818
9,Control,Subject011,-5,0.054012,-0.055642,0.182810,0.491711,4.371947



Summary by state:


median_ROI_r                               FC_matrix_r            \
              median      mean       min       max      median      mean   
state                                                                      
Control     0.022477  0.025519 -0.001961  0.054012    0.094265  0.098531   
SDV         0.030552  0.031757 -0.001577  0.075706    0.127113  0.107034   

                            median_std_ratio_fmriprep_over_CONN            \
              min       max                              median      mean   
state                                                                       
Control -0.081942  0.491711                            2.903475  2.936851   
SDV     -0.117037  0.326042                            2.868354  2.915255   

                             
              min       max  
state                        
Control  2.083066  4.371947  
SDV      1.982850  3.947434